# EXP21D-v8 — EXP 1B Equal-Dimension PCA Fair Control
PCA 128-d → 10-d is fit on train only. XGBoost compares tabular 10-d, GNN-PCA 10-d, and hybrid 20-d for all corrected samplers. Attach the completed EXP21D-v5 kernel output before running. This notebook is not pushed automatically.


In [ ]:
%%writefile requirements.txt
numpy>=1.26,<3
pandas>=2.1,<4
scikit-learn>=1.4,<2
xgboost>=2.0,<4
matplotlib>=3.8,<4
PyYAML>=6,<7
joblib>=1.3,<2



In [ ]:
%%writefile config.kaggle.yaml
experiment:
  id: "EXP21D-v8-FairCtrl-1B"
  seed: 42
  evaluation_seed: 1042
  samplers: ["uniform", "topology", "importance"]

input:
  source_result_dir: "/kaggle/input/exp21d-v5-sampler-correctness/result/exp21d_v5_sampler_correctness/seed_42"
  expected_hybrid_dimensions: 138
  tabular_dimensions: 10
  embedding_dimensions: 128
  compute_sha256: false

pca: {n_components: 10, batch_size: 65536, transform_batch_size: 100000, whiten: false}
classifier: {n_estimators: 100, max_depth: 4, learning_rate: 0.05, min_child_weight: 10, subsample: 0.8, colsample_bytree: 0.8, scale_pos_weight_strategy: "sqrt", tree_method: "hist", n_jobs: -1}
threshold: {min_precision: 0.15}
permutation_importance: {repeats: 20, blocks: ["tabular_10d", "gnn_pca_10d"]}
output: {result_dir: "/kaggle/working/result/exp21d_v8_fairctrl_1b", save_transformed_features: true, snapshot_source: true}
smoke: {max_rows_per_split: 5000, pca_batch_size: 512, permutation_repeats: 2, n_estimators: 10}



In [ ]:
%%writefile config.py
from __future__ import annotations

from pathlib import Path

import yaml


def load_config(path: str | Path) -> dict:
    source = Path(path)
    if not source.is_file():
        raise FileNotFoundError(f"Config tidak ditemukan: {source}")
    config = yaml.safe_load(source.read_text(encoding="utf-8"))
    required = {
        "experiment", "input", "pca", "classifier", "threshold",
        "permutation_importance", "output", "smoke",
    }
    missing = sorted(required - set(config))
    if missing:
        raise ValueError(f"Bagian config belum lengkap: {missing}")
    if int(config["pca"]["n_components"]) != int(config["input"]["tabular_dimensions"]):
        raise ValueError("EXP 1B mensyaratkan dimensi PCA sama dengan dimensi tabular")
    return config



In [ ]:
%%writefile artifacts.py
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


def write_json(path: str | Path, value: Any) -> None:
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_suffix(target.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, indent=2, ensure_ascii=False, default=str) + "\n",
        encoding="utf-8",
    )
    temporary.replace(target)


class RunLogger:
    """Append-only event history so an interrupted run remains inspectable."""

    def __init__(self, path: str | Path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)

    def log(self, event: str, **payload: Any) -> None:
        record = {
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "event": event,
            **payload,
        }
        with self.path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")


def build_artifact_index(run_dir: str | Path) -> dict[str, Any]:
    root = Path(run_dir)
    files = []
    for item in sorted(root.rglob("*")):
        if item.is_file() and item.name != "artifact_index.json":
            files.append({"path": item.relative_to(root).as_posix(), "size_bytes": item.stat().st_size})
    result = {"run_dir": str(root.resolve()), "file_count": len(files), "files": files}
    write_json(root / "artifact_index.json", result)
    return result



In [ ]:
%%writefile reproducibility.py
from __future__ import annotations

import hashlib
import json
import os
import platform
import random
import shutil
import subprocess
import sys
from pathlib import Path
from typing import Any

import numpy as np


def stable_hash(value: Any, length: int = 12) -> str:
    payload = json.dumps(value, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()[:length]


def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)


def file_fingerprint(path: str | Path, with_sha256: bool = False) -> dict[str, Any]:
    source = Path(path).resolve()
    stat = source.stat()
    result = {
        "path": str(source),
        "size_bytes": stat.st_size,
        "modified_time_ns": stat.st_mtime_ns,
    }
    if with_sha256:
        digest = hashlib.sha256()
        with source.open("rb") as handle:
            while block := handle.read(8 * 1024 * 1024):
                digest.update(block)
        result["sha256"] = digest.hexdigest()
    return result


def git_revision(path: str | Path) -> dict[str, Any]:
    try:
        commit = subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=path, text=True, stderr=subprocess.DEVNULL
        ).strip()
        dirty = bool(subprocess.check_output(["git", "status", "--porcelain"], cwd=path, text=True).strip())
        return {"commit": commit, "dirty": dirty}
    except (OSError, subprocess.CalledProcessError):
        return {"commit": None, "dirty": None}


def environment_manifest() -> dict[str, Any]:
    import pandas
    import sklearn
    import xgboost

    return {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pandas.__version__,
        "scikit_learn": sklearn.__version__,
        "xgboost": xgboost.__version__,
    }


def snapshot_source(source_dir: str | Path, destination: str | Path) -> None:
    source, target = Path(source_dir), Path(destination)
    target.mkdir(parents=True, exist_ok=True)
    for item in source.iterdir():
        if item.is_file() and item.suffix.lower() in {".py", ".yaml", ".yml", ".md", ".txt"}:
            shutil.copy2(item, target / item.name)



In [ ]:
%%writefile metrics.py
from __future__ import annotations

from typing import Any

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)


def find_optimal_threshold(
    probabilities: np.ndarray, labels: np.ndarray, min_precision: float = 0.15
) -> tuple[float, dict[str, Any]]:
    precision, recall, thresholds = precision_recall_curve(labels, probabilities)
    if thresholds.size == 0:
        return 0.5, {"tau": 0.5, "val_f1": 0.0, "candidate_thresholds": 0}
    p, r = precision[:-1], recall[:-1]
    f1 = np.divide(2 * p * r, p + r, out=np.zeros_like(p), where=(p + r) > 0)
    eligible = p >= min_precision
    indices = np.flatnonzero(eligible) if eligible.any() else np.arange(len(thresholds))
    best = int(indices[np.argmax(f1[indices])])
    return float(thresholds[best]), {
        "tau": float(thresholds[best]),
        "val_f1": float(f1[best]),
        "val_precision": float(p[best]),
        "val_recall": float(r[best]),
        "candidate_thresholds": int(len(thresholds)),
        "min_precision_constraint": float(min_precision),
        "constraint_applied": bool(eligible.any()),
    }


def evaluate_predictions(probabilities: np.ndarray, labels: np.ndarray, threshold: float) -> dict[str, Any]:
    predictions = (probabilities >= threshold).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    recall = float(recall_score(labels, predictions, zero_division=0))
    precision = float(precision_score(labels, predictions, zero_division=0))
    specificity = float(tn / (tn + fp)) if tn + fp else 0.0
    return {
        "threshold": float(threshold),
        "f1": float(f1_score(labels, predictions, zero_division=0)),
        "recall": recall,
        "precision": precision,
        "accuracy": float(accuracy_score(labels, predictions)),
        "gmean": float(np.sqrt(recall * specificity)),
        "auprc": float(average_precision_score(labels, probabilities)),
        "roc_auc": float(roc_auc_score(labels, probabilities)),
        "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    }


def threshold_curve(probabilities: np.ndarray, labels: np.ndarray, points: int = 181) -> dict[str, list[float]]:
    thresholds = np.linspace(0.05, 0.95, points)
    f1_values, recall_values, precision_values = [], [], []
    for threshold in thresholds:
        prediction = probabilities >= threshold
        f1_values.append(float(f1_score(labels, prediction, zero_division=0)))
        recall_values.append(float(recall_score(labels, prediction, zero_division=0)))
        precision_values.append(float(precision_score(labels, prediction, zero_division=0)))
    return {
        "threshold": thresholds.tolist(),
        "f1": f1_values,
        "recall": recall_values,
        "precision": precision_values,
    }


In [ ]:
%%writefile pca_pipeline.py
from __future__ import annotations

import json
from pathlib import Path
from typing import Iterator

import numpy as np
from sklearn.decomposition import IncrementalPCA


SPLITS = ("train", "validation", "test")


def resolve_source_result_dir(value: str, config_path: Path) -> Path:
    configured = Path(value)
    if not configured.is_absolute():
        configured = (config_path.parent / configured).resolve()
    if (configured / "exp21d_v5_master_comparison.csv").is_file():
        return configured
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        matches = sorted(kaggle_input.rglob("exp21d_v5_master_comparison.csv"))
        if len(matches) == 1:
            return matches[0].parent
        if len(matches) > 1:
            raise RuntimeError(
                "Sumber EXP 2A ambigu: " + ", ".join(str(item.parent) for item in matches)
            )
    raise FileNotFoundError(f"Direktori hasil EXP 2A tidak ditemukan: {configured}")


def validate_source_contract(
    source_dir: Path,
    samplers: list[str],
    expected_dimensions: int,
    tabular_dimensions: int,
    embedding_dimensions: int,
) -> dict[str, dict[str, object]]:
    if tabular_dimensions + embedding_dimensions != expected_dimensions:
        raise ValueError("Dimensi tabular dan embedding tidak menjumlah ke dimensi hybrid")
    contracts: dict[str, dict[str, object]] = {}
    reference_shapes: dict[str, tuple[int, ...]] | None = None
    for sampler in samplers:
        feature_dir = source_dir / "features" / sampler
        contract_path = feature_dir / "feature_contract.json"
        if not contract_path.is_file():
            raise FileNotFoundError(f"Feature contract tidak ditemukan: {contract_path}")
        contract = json.loads(contract_path.read_text(encoding="utf-8"))
        if int(contract.get("hybrid_dimensions", -1)) != expected_dimensions:
            raise RuntimeError(f"Dimensi hybrid {sampler} tidak sesuai: {contract}")
        if int(contract.get("tabular_dimensions", -1)) != tabular_dimensions:
            raise RuntimeError(f"Dimensi tabular {sampler} tidak sesuai: {contract}")
        if int(contract.get("graphsage_embedding_dimensions", -1)) != embedding_dimensions:
            raise RuntimeError(f"Dimensi embedding {sampler} tidak sesuai: {contract}")
        shapes: dict[str, tuple[int, ...]] = {}
        positives: dict[str, int] = {}
        for split in SPLITS:
            feature_path = feature_dir / f"{split}_hybrid_features.npy"
            label_path = feature_dir / f"{split}_labels.npy"
            if not feature_path.is_file() or not label_path.is_file():
                raise FileNotFoundError(f"Artefak {sampler}/{split} belum lengkap")
            features = np.load(feature_path, mmap_mode="r")
            labels = np.load(label_path, mmap_mode="r")
            if features.ndim != 2 or features.shape[1] != expected_dimensions:
                raise RuntimeError(f"Shape fitur {sampler}/{split} tidak sesuai: {features.shape}")
            if labels.shape != (features.shape[0],):
                raise RuntimeError(f"Shape label {sampler}/{split} tidak sesuai: {labels.shape}")
            unique = np.unique(labels)
            if not set(unique.tolist()).issubset({0, 1}):
                raise RuntimeError(f"Label {sampler}/{split} bukan biner: {unique.tolist()}")
            shapes[split] = tuple(features.shape)
            positives[split] = int(labels.sum())
        if reference_shapes is None:
            reference_shapes = shapes
        elif shapes != reference_shapes:
            raise RuntimeError(f"Shape split tidak identik antar-sampler: {sampler}={shapes}")
        contracts[sampler] = {"contract": contract, "shapes": shapes, "positives": positives}
    return contracts


def load_sampler_arrays(source_dir: Path, sampler: str, max_rows: int | None = None) -> tuple[dict, dict]:
    feature_dir = source_dir / "features" / sampler
    features, labels = {}, {}
    for split in SPLITS:
        matrix = np.load(feature_dir / f"{split}_hybrid_features.npy", mmap_mode="r")
        target = np.load(feature_dir / f"{split}_labels.npy", mmap_mode="r")
        if max_rows is not None:
            limit = min(max_rows, len(matrix))
            matrix, target = matrix[:limit], target[:limit]
        features[split], labels[split] = matrix, target
    return features, labels


def batch_ranges(total: int, batch_size: int, minimum_last_batch: int) -> Iterator[tuple[int, int]]:
    if total < minimum_last_batch:
        raise ValueError(f"Jumlah baris {total} lebih kecil dari minimum {minimum_last_batch}")
    start = 0
    while start < total:
        stop = min(start + batch_size, total)
        if 0 < total - stop < minimum_last_batch:
            stop = total
        yield start, stop
        start = stop


def fit_incremental_pca(
    train_hybrid: np.ndarray,
    tabular_dimensions: int,
    n_components: int,
    batch_size: int,
    whiten: bool,
) -> IncrementalPCA:
    if train_hybrid.shape[1] <= tabular_dimensions:
        raise ValueError("Matriks input tidak memiliki blok embedding")
    model = IncrementalPCA(n_components=n_components, whiten=whiten, batch_size=batch_size)
    for start, stop in batch_ranges(len(train_hybrid), batch_size, n_components):
        batch = np.asarray(train_hybrid[start:stop, tabular_dimensions:], dtype=np.float32)
        if not np.isfinite(batch).all():
            raise ValueError(f"Embedding train mengandung NaN/Inf pada baris {start}:{stop}")
        model.partial_fit(batch)
    return model


def transform_embedding_to_file(
    model: IncrementalPCA,
    hybrid: np.ndarray,
    tabular_dimensions: int,
    output_path: Path,
    batch_size: int,
) -> Path:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output = np.lib.format.open_memmap(
        output_path, mode="w+", dtype=np.float32, shape=(len(hybrid), model.n_components_)
    )
    for start in range(0, len(hybrid), batch_size):
        stop = min(start + batch_size, len(hybrid))
        transformed = model.transform(np.asarray(hybrid[start:stop, tabular_dimensions:], dtype=np.float32))
        output[start:stop] = transformed.astype(np.float32, copy=False)
    output.flush()
    del output
    return output_path


def create_hybrid_20d_file(
    source_hybrid: np.ndarray,
    pca_path: Path,
    tabular_dimensions: int,
    output_path: Path,
    batch_size: int,
) -> Path:
    pca_features = np.load(pca_path, mmap_mode="r")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output = np.lib.format.open_memmap(
        output_path,
        mode="w+",
        dtype=np.float32,
        shape=(len(source_hybrid), tabular_dimensions + pca_features.shape[1]),
    )
    for start in range(0, len(source_hybrid), batch_size):
        stop = min(start + batch_size, len(source_hybrid))
        output[start:stop, :tabular_dimensions] = source_hybrid[start:stop, :tabular_dimensions]
        output[start:stop, tabular_dimensions:] = pca_features[start:stop]
    output.flush()
    del output, pca_features
    return output_path


def permuted_block(
    matrix: np.ndarray, start: int, stop: int, permutation: np.ndarray
) -> np.ndarray:
    if len(permutation) != len(matrix):
        raise ValueError("Panjang permutation tidak sama dengan jumlah baris")
    output = np.array(matrix, dtype=np.float32, copy=True)
    output[:, start:stop] = matrix[permutation, start:stop]
    return output


In [ ]:
%%writefile visualize_results.py
from __future__ import annotations

import json
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve


SAMPLER_COLORS = {"uniform": "#4C78A8", "topology": "#59A14F", "importance": "#E15759"}
REPRESENTATION_COLORS = {
    "tabular_10d": "#F28E2B",
    "gnn_pca_10d": "#76B7B2",
    "hybrid_20d": "#4E79A7",
}


def setup_style() -> None:
    plt.rcParams.update(
        {
            "font.size": 10,
            "axes.titlesize": 12,
            "axes.labelsize": 11,
            "legend.fontsize": 9,
            "figure.dpi": 140,
            "savefig.dpi": 300,
            "savefig.bbox": "tight",
        }
    )


def save_figure(fig: plt.Figure, output_dir: Path, stem: str) -> None:
    fig.savefig(output_dir / f"{stem}.png", dpi=300, bbox_inches="tight")
    fig.savefig(output_dir / f"{stem}.pdf", bbox_inches="tight")
    plt.close(fig)


def plot_equal_dimension_metrics(run_dir: Path, output_dir: Path, data_dir: Path) -> None:
    frame = pd.read_csv(run_dir / "equal_dimension_comparison.csv")
    frame.to_csv(data_dir / "equal_dimension_metrics.csv", index=False)
    order = ["tabular_10d", "gnn_pca_10d", "hybrid_20d"]
    samplers = list(frame["sampler"].drop_duplicates())
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharex=True)
    width = 0.24
    x = np.arange(len(samplers))
    for offset, representation in enumerate(order):
        subset = frame[frame["representation"] == representation].set_index("sampler").loc[samplers]
        for axis, metric, title in zip(axes, ("test_f1", "test_auprc"), ("F1-Score", "AUPRC")):
            bars = axis.bar(
                x + (offset - 1) * width,
                subset[metric],
                width,
                label=representation.replace("_", " "),
                color=REPRESENTATION_COLORS[representation],
                edgecolor="black",
                linewidth=0.5,
            )
            axis.bar_label(bars, fmt="%.3f", padding=2, fontsize=8, rotation=90)
            axis.set_title(f"Equal-Dimension Comparison: {title}", fontweight="bold")
            axis.set_ylabel(title)
            axis.grid(axis="y", linestyle="--", alpha=0.4)
            axis.set_ylim(max(0.0, float(frame[metric].min()) - 0.08), 1.015)
    for axis in axes:
        axis.set_xticks(x, [item.capitalize() for item in samplers])
    axes[0].legend(loc="lower left")
    fig.suptitle("EXP 1B — 10-d Tabular vs 10-d GNN-PCA vs Hybrid 20-d", fontweight="bold")
    fig.tight_layout()
    save_figure(fig, output_dir, "exp21d_v8_equal_dimension_metrics")


def plot_lift_from_tabular(run_dir: Path, output_dir: Path, data_dir: Path) -> None:
    frame = pd.read_csv(run_dir / "equal_dimension_comparison.csv")
    records = []
    for sampler, group in frame.groupby("sampler"):
        baseline = group.set_index("representation").loc["tabular_10d"]
        for representation in ("gnn_pca_10d", "hybrid_20d"):
            row = group.set_index("representation").loc[representation]
            records.append(
                {
                    "sampler": sampler,
                    "representation": representation,
                    "delta_f1": row["test_f1"] - baseline["test_f1"],
                    "delta_auprc": row["test_auprc"] - baseline["test_auprc"],
                }
            )
    lift = pd.DataFrame(records)
    lift.to_csv(data_dir / "lift_from_tabular.csv", index=False)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    width = 0.36
    samplers = list(lift["sampler"].drop_duplicates())
    x = np.arange(len(samplers))
    for index, representation in enumerate(("gnn_pca_10d", "hybrid_20d")):
        subset = lift[lift["representation"] == representation].set_index("sampler").loc[samplers]
        for axis, metric, title in zip(axes, ("delta_f1", "delta_auprc"), ("ΔF1", "ΔAUPRC")):
            bars = axis.bar(
                x + (index - 0.5) * width,
                subset[metric],
                width,
                label=representation.replace("_", " "),
                color=REPRESENTATION_COLORS[representation],
                edgecolor="black",
                linewidth=0.5,
            )
            axis.bar_label(bars, fmt="%+.4f", padding=3, fontsize=8)
            axis.axhline(0, color="black", linewidth=0.8)
            axis.set_title(title + " terhadap Tabular 10-d", fontweight="bold")
            axis.grid(axis="y", linestyle="--", alpha=0.4)
            axis.set_xticks(x, [item.capitalize() for item in samplers])
    axes[0].legend()
    fig.tight_layout()
    save_figure(fig, output_dir, "exp21d_v8_lift_from_tabular")


def plot_pca_variance(run_dir: Path, output_dir: Path, data_dir: Path) -> None:
    records = []
    history_root = run_dir / "history"
    for path in sorted(history_root.glob("*/pca_summary.json")):
        payload = json.loads(path.read_text(encoding="utf-8"))
        cumulative = payload["cumulative_explained_variance"]
        individual = payload["explained_variance_ratio"]
        for index, (ratio, cumulative_ratio) in enumerate(zip(individual, cumulative), start=1):
            records.append(
                {
                    "sampler": payload["sampler"],
                    "component": index,
                    "explained_variance_ratio": ratio,
                    "cumulative_explained_variance": cumulative_ratio,
                }
            )
    frame = pd.DataFrame(records)
    frame.to_csv(data_dir / "pca_explained_variance.csv", index=False)
    fig, ax = plt.subplots(figsize=(9, 5.5))
    for sampler, group in frame.groupby("sampler"):
        ax.plot(
            group["component"], group["cumulative_explained_variance"], marker="o",
            linewidth=2, label=sampler.capitalize(), color=SAMPLER_COLORS.get(sampler),
        )
    ax.set_xticks(range(1, 11))
    ax.set_xlabel("Jumlah komponen PCA")
    ax.set_ylabel("Cumulative explained variance ratio")
    ax.set_title("Retensi Varians Embedding GraphSAGE 128-d dalam PCA 10-d", fontweight="bold")
    ax.grid(linestyle="--", alpha=0.4)
    ax.legend()
    fig.tight_layout()
    save_figure(fig, output_dir, "exp21d_v8_pca_explained_variance")


def plot_permutation_importance(run_dir: Path, output_dir: Path, data_dir: Path) -> None:
    frame = pd.read_csv(run_dir / "block_permutation_importance_repeats.csv")
    frame.to_csv(data_dir / "block_permutation_importance_repeats.csv", index=False)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    positions, labels, colors = [], [], []
    auprc_values, f1_values = [], []
    position = 1
    for sampler in frame["sampler"].drop_duplicates():
        for block in ("tabular_10d", "gnn_pca_10d"):
            subset = frame[(frame["sampler"] == sampler) & (frame["block"] == block)]
            positions.append(position)
            labels.append(f"{sampler.capitalize()}\n{block.replace('_', ' ')}")
            colors.append(REPRESENTATION_COLORS[block])
            auprc_values.append(subset["drop_auprc"].to_numpy())
            f1_values.append(subset["drop_f1"].to_numpy())
            position += 1
        position += 0.5
    for axis, values, title in zip(axes, (auprc_values, f1_values), ("Drop in AUPRC", "Drop in F1")):
        boxes = axis.boxplot(values, positions=positions, widths=0.55, patch_artist=True, showmeans=True)
        for patch, color in zip(boxes["boxes"], colors):
            patch.set_facecolor(color)
        axis.axhline(0, color="black", linewidth=0.8)
        axis.set_xticks(positions, labels, rotation=35, ha="right")
        axis.set_title(title + " setelah Block Permutation", fontweight="bold")
        axis.grid(axis="y", linestyle="--", alpha=0.4)
    fig.tight_layout()
    save_figure(fig, output_dir, "exp21d_v8_block_permutation_importance")


def plot_precision_recall(run_dir: Path, output_dir: Path, data_dir: Path) -> None:
    summary = pd.read_csv(run_dir / "equal_dimension_comparison.csv")
    samplers = list(summary["sampler"].drop_duplicates())
    fig, axes = plt.subplots(1, len(samplers), figsize=(6 * len(samplers), 5.2), sharey=True)
    if len(samplers) == 1:
        axes = [axes]
    source_records = []
    for axis, sampler in zip(axes, samplers):
        archive = np.load(run_dir / "predictions" / f"{sampler}.npz")
        labels = archive["test_labels"]
        for representation in ("tabular_10d", "gnn_pca_10d", "hybrid_20d"):
            probabilities = archive[f"{representation}_test_probabilities"]
            precision, recall, _ = precision_recall_curve(labels, probabilities)
            metric = summary[
                (summary["sampler"] == sampler) & (summary["representation"] == representation)
            ].iloc[0]
            axis.plot(
                recall, precision, linewidth=2,
                color=REPRESENTATION_COLORS[representation],
                label=f"{representation.replace('_', ' ')} ({metric['test_auprc']:.4f})",
            )
            sample_index = np.linspace(0, len(recall) - 1, min(1000, len(recall)), dtype=int)
            source_records.extend(
                {
                    "sampler": sampler,
                    "representation": representation,
                    "recall": float(recall[index]),
                    "precision": float(precision[index]),
                }
                for index in sample_index
            )
        axis.set_title(sampler.capitalize(), fontweight="bold")
        axis.set_xlabel("Recall")
        axis.grid(linestyle="--", alpha=0.4)
        axis.legend(loc="lower left")
    axes[0].set_ylabel("Precision")
    fig.suptitle("Precision-Recall Curves — Equal-Dimension Fair Control", fontweight="bold")
    fig.tight_layout()
    pd.DataFrame(source_records).to_csv(data_dir / "precision_recall_curve_points.csv", index=False)
    save_figure(fig, output_dir, "exp21d_v8_precision_recall_curves")


def create_all_visualizations(run_dir: str | Path) -> None:
    root = Path(run_dir)
    output_dir = root / "plots"
    data_dir = output_dir / "source_data"
    output_dir.mkdir(parents=True, exist_ok=True)
    data_dir.mkdir(parents=True, exist_ok=True)
    setup_style()
    plot_equal_dimension_metrics(root, output_dir, data_dir)
    plot_lift_from_tabular(root, output_dir, data_dir)
    plot_pca_variance(root, output_dir, data_dir)
    plot_permutation_importance(root, output_dir, data_dir)
    plot_precision_recall(root, output_dir, data_dir)


In [ ]:
%%writefile main.py
from __future__ import annotations

import argparse
import gc
import json
import time
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
import yaml
from sklearn.metrics import average_precision_score, f1_score
from xgboost import XGBClassifier

from artifacts import RunLogger, build_artifact_index, write_json
from config import load_config
from metrics import evaluate_predictions, find_optimal_threshold, threshold_curve
from pca_pipeline import (
    SPLITS,
    create_hybrid_20d_file,
    fit_incremental_pca,
    load_sampler_arrays,
    permuted_block,
    resolve_source_result_dir,
    transform_embedding_to_file,
    validate_source_contract,
)
from reproducibility import (
    environment_manifest,
    file_fingerprint,
    git_revision,
    seed_everything,
    snapshot_source,
    stable_hash,
)
from visualize_results import create_all_visualizations


REPRESENTATIONS = {
    "tabular_10d": {"display_name": "Clean Tabular 10-d", "dimensions": 10},
    "gnn_pca_10d": {"display_name": "GraphSAGE PCA 10-d", "dimensions": 10},
    "hybrid_20d": {"display_name": "Tabular 10-d + GraphSAGE PCA 10-d", "dimensions": 20},
}


def resolve_output_path(value: str, config_path: Path) -> Path:
    path = Path(value)
    return path if path.is_absolute() else (config_path.parent / path).resolve()


def xgb_parameters(config: dict, labels: np.ndarray, seed: int, smoke: bool, smoke_cfg: dict) -> dict[str, Any]:
    positives = int(labels.sum())
    negatives = int(len(labels) - positives)
    ratio = negatives / max(positives, 1)
    strategy = config.get("scale_pos_weight_strategy", "sqrt")
    scale = np.sqrt(ratio) if strategy == "sqrt" else ratio if strategy == "full" else 1.0
    estimators = int(smoke_cfg["n_estimators"] if smoke else config["n_estimators"])
    return {
        "n_estimators": estimators,
        "max_depth": int(config["max_depth"]),
        "learning_rate": float(config["learning_rate"]),
        "min_child_weight": float(config["min_child_weight"]),
        "subsample": float(config["subsample"]),
        "colsample_bytree": float(config["colsample_bytree"]),
        "scale_pos_weight": float(scale),
        "tree_method": config.get("tree_method", "hist"),
        "random_state": int(seed),
        "n_jobs": int(config.get("n_jobs", -1)),
        "eval_metric": "logloss",
    }


def select_smoke_rows(features: dict, labels: dict, limit: int, seed: int) -> tuple[dict, dict]:
    selected_features, selected_labels = {}, {}
    for offset, split in enumerate(SPLITS):
        target = np.asarray(labels[split])
        if len(target) <= limit:
            indices = np.arange(len(target))
        else:
            rng = np.random.default_rng(seed + offset)
            positive = np.flatnonzero(target == 1)
            negative = np.flatnonzero(target == 0)
            positive_take = min(len(positive), max(2, limit // 10))
            chosen_positive = rng.choice(positive, positive_take, replace=False) if len(positive) else np.array([], dtype=int)
            negative_take = limit - len(chosen_positive)
            chosen_negative = rng.choice(negative, negative_take, replace=False)
            indices = np.sort(np.concatenate([chosen_positive, chosen_negative]))
        selected_features[split] = np.asarray(features[split][indices], dtype=np.float32)
        selected_labels[split] = np.asarray(target[indices], dtype=np.int8)
        if np.unique(selected_labels[split]).size != 2:
            raise RuntimeError(f"Smoke split {split} tidak memiliki dua kelas")
    return selected_features, selected_labels


def verify_cross_sampler_invariants(source_dir: Path, samplers: list[str], chunk_size: int = 100_000) -> dict:
    reference = samplers[0]
    checks = []
    for sampler in samplers[1:]:
        for split in SPLITS:
            ref_dir = source_dir / "features" / reference
            cur_dir = source_dir / "features" / sampler
            ref_labels = np.load(ref_dir / f"{split}_labels.npy", mmap_mode="r")
            cur_labels = np.load(cur_dir / f"{split}_labels.npy", mmap_mode="r")
            if not np.array_equal(ref_labels, cur_labels):
                raise RuntimeError(f"Label {split} berbeda antara {reference} dan {sampler}")
            ref_features = np.load(ref_dir / f"{split}_hybrid_features.npy", mmap_mode="r")
            cur_features = np.load(cur_dir / f"{split}_hybrid_features.npy", mmap_mode="r")
            for start in range(0, len(ref_features), chunk_size):
                stop = min(start + chunk_size, len(ref_features))
                if not np.array_equal(ref_features[start:stop, :10], cur_features[start:stop, :10]):
                    raise RuntimeError(f"Blok tabular {split} berbeda antara {reference} dan {sampler}")
            checks.append({"sampler": sampler, "split": split, "labels_equal": True, "tabular_equal": True})
    return {"reference_sampler": reference, "status": "passed", "checks": checks}


def train_and_evaluate(
    sampler: str,
    representation: str,
    matrices: dict[str, np.ndarray],
    labels: dict[str, np.ndarray],
    cfg: dict,
    seed: int,
    smoke: bool,
    run_dir: Path,
) -> tuple[XGBClassifier, dict[str, Any], dict[str, np.ndarray]]:
    parameters = xgb_parameters(cfg["classifier"], labels["train"], seed, smoke, cfg["smoke"])
    model = XGBClassifier(**parameters)
    started = time.time()
    model.fit(
        matrices["train"],
        labels["train"],
        eval_set=[(matrices["validation"], labels["validation"])],
        verbose=False,
    )
    validation_probabilities = model.predict_proba(matrices["validation"])[:, 1].astype(np.float32)
    test_probabilities = model.predict_proba(matrices["test"])[:, 1].astype(np.float32)
    threshold, selection = find_optimal_threshold(
        validation_probabilities,
        labels["validation"],
        float(cfg["threshold"]["min_precision"]),
    )
    result = {
        "sampler": sampler,
        "representation": representation,
        **REPRESENTATIONS[representation],
        "xgboost_parameters": parameters,
        "threshold_selection": selection,
        "validation": evaluate_predictions(validation_probabilities, labels["validation"], threshold),
        "test": evaluate_predictions(test_probabilities, labels["test"], threshold),
        "threshold_curve_test": threshold_curve(test_probabilities, labels["test"]),
        "fit_and_predict_seconds": time.time() - started,
    }
    model_dir = run_dir / "models" / sampler
    model_dir.mkdir(parents=True, exist_ok=True)
    model.save_model(model_dir / f"xgboost_{representation}.json")
    write_json(run_dir / "history" / sampler / f"xgboost_{representation}.json", model.evals_result())
    probabilities = {
        "validation": validation_probabilities,
        "test": test_probabilities,
        "threshold": np.asarray(threshold, dtype=np.float64),
    }
    return model, result, probabilities


def calculate_block_permutation_importance(
    model: XGBClassifier,
    test_matrix: np.ndarray,
    test_labels: np.ndarray,
    threshold: float,
    sampler: str,
    repeats: int,
    seed: int,
) -> list[dict[str, Any]]:
    baseline_probability = model.predict_proba(test_matrix)[:, 1]
    baseline_auprc = float(average_precision_score(test_labels, baseline_probability))
    baseline_f1 = float(f1_score(test_labels, baseline_probability >= threshold, zero_division=0))
    blocks = {"tabular_10d": (0, 10), "gnn_pca_10d": (10, 20)}
    records = []
    for repeat in range(repeats):
        rng = np.random.default_rng(seed + repeat)
        permutation = rng.permutation(len(test_matrix))
        for block, (start, stop) in blocks.items():
            permuted = permuted_block(test_matrix, start, stop, permutation)
            probability = model.predict_proba(permuted)[:, 1]
            permuted_auprc = float(average_precision_score(test_labels, probability))
            permuted_f1 = float(f1_score(test_labels, probability >= threshold, zero_division=0))
            records.append(
                {
                    "sampler": sampler,
                    "repeat": repeat,
                    "permutation_seed": seed + repeat,
                    "block": block,
                    "baseline_auprc": baseline_auprc,
                    "permuted_auprc": permuted_auprc,
                    "drop_auprc": baseline_auprc - permuted_auprc,
                    "baseline_f1": baseline_f1,
                    "permuted_f1": permuted_f1,
                    "drop_f1": baseline_f1 - permuted_f1,
                }
            )
            del permuted, probability
    return records


def result_row(result: dict[str, Any]) -> dict[str, Any]:
    test = result["test"]
    cm = test["confusion_matrix"]
    return {
        "sampler": result["sampler"],
        "representation": result["representation"],
        "display_name": result["display_name"],
        "dimensions": result["dimensions"],
        "optimal_threshold": test["threshold"],
        "test_f1": test["f1"],
        "test_recall": test["recall"],
        "test_precision": test["precision"],
        "test_auprc": test["auprc"],
        "test_roc_auc": test["roc_auc"],
        "false_positive": cm["fp"],
        "false_negative": cm["fn"],
    }


def dataframe_to_markdown(frame: pd.DataFrame, decimals: int = 6) -> str:
    def render(value: Any) -> str:
        if isinstance(value, (float, np.floating)):
            return f"{float(value):.{decimals}f}"
        return str(value).replace("|", "\\|")

    columns = [str(column) for column in frame.columns]
    lines = [
        "| " + " | ".join(columns) + " |",
        "| " + " | ".join("---" for _ in columns) + " |",
    ]
    for row in frame.itertuples(index=False, name=None):
        lines.append("| " + " | ".join(render(value) for value in row) + " |")
    return "\n".join(lines)


def write_summary(run_dir: Path, rows: pd.DataFrame, permutation: pd.DataFrame) -> None:
    best_f1 = rows.loc[rows["test_f1"].idxmax()]
    best_auprc = rows.loc[rows["test_auprc"].idxmax()]
    aggregate = permutation.groupby(["sampler", "block"], as_index=False).agg(
        drop_auprc_mean=("drop_auprc", "mean"),
        drop_auprc_std=("drop_auprc", "std"),
        drop_f1_mean=("drop_f1", "mean"),
        drop_f1_std=("drop_f1", "std"),
    )
    text = [
        "# EXP21D-v8 Fair Control 1B\n",
        "PCA di-fit hanya pada train; threshold dipilih hanya pada validation.\n",
        "## Equal-dimension comparison\n",
        dataframe_to_markdown(rows),
        "\n## Block permutation importance pada hybrid 20-d\n",
        dataframe_to_markdown(aggregate),
        "\n## Ringkasan\n",
        f"- F1 tertinggi: {best_f1['sampler']} / {best_f1['representation']} = {best_f1['test_f1']:.6f}\n",
        f"- AUPRC tertinggi: {best_auprc['sampler']} / {best_auprc['representation']} = {best_auprc['test_auprc']:.6f}\n",
        "- Hasil single-seed belum merupakan bukti signifikansi statistik lintas seed.\n",
    ]
    (run_dir / "summary_report.md").write_text("\n".join(text), encoding="utf-8")


def run_experiment(args: argparse.Namespace) -> Path:
    config_path = Path(args.config).resolve()
    cfg = load_config(config_path)
    seed = int(args.seed if args.seed is not None else cfg["experiment"]["seed"])
    evaluation_seed = int(cfg["experiment"].get("evaluation_seed", seed + 1000))
    seed_everything(seed)
    samplers = list(cfg["experiment"]["samplers"])
    source_dir = resolve_source_result_dir(cfg["input"]["source_result_dir"], config_path)
    output_root = resolve_output_path(cfg["output"]["result_dir"], config_path)
    config_hash = stable_hash({"config": cfg, "source": str(source_dir), "smoke": args.smoke})
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_id = args.run_id or f"exp21d_v8_fairctrl_1b_seed{seed}_{config_hash}_{stamp}"
    run_dir = output_root / ("smoke" if args.smoke else "runs") / run_id
    run_dir.mkdir(parents=True, exist_ok=False)
    args._active_run_dir = run_dir
    logger = RunLogger(run_dir / "events.jsonl")
    manifest = {
        "experiment_id": cfg["experiment"]["id"],
        "run_id": run_id,
        "status": "running",
        "scientific_status": "smoke_only_not_for_paper" if args.smoke else "paper_candidate_single_seed",
        "seed": seed,
        "evaluation_seed": evaluation_seed,
        "source_result_dir": str(source_dir.resolve()),
        "config_hash": config_hash,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "git": git_revision(Path(__file__).resolve().parent),
    }
    write_json(run_dir / "manifest.json", manifest)
    logger.log("run_started", run_id=run_id, smoke=args.smoke)
    if cfg["output"].get("snapshot_source", True):
        snapshot_source(Path(__file__).resolve().parent, run_dir / "code_snapshot")
    (run_dir / "resolved_config.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    write_json(run_dir / "environment.json", environment_manifest())
    started = time.time()

    contracts = validate_source_contract(
        source_dir,
        samplers,
        int(cfg["input"]["expected_hybrid_dimensions"]),
        int(cfg["input"]["tabular_dimensions"]),
        int(cfg["input"]["embedding_dimensions"]),
    )
    write_json(run_dir / "source_contracts.json", contracts)
    invariant_audit = verify_cross_sampler_invariants(source_dir, samplers)
    write_json(run_dir / "cross_sampler_invariant_audit.json", invariant_audit)
    manifest["source_master_comparison"] = file_fingerprint(
        source_dir / "exp21d_v5_master_comparison.csv",
        bool(cfg["input"].get("compute_sha256", False)),
    )
    write_json(run_dir / "manifest.json", manifest)
    logger.log("source_validated", samplers=samplers, invariant_status=invariant_audit["status"])

    all_rows: list[dict[str, Any]] = []
    all_results: list[dict[str, Any]] = []
    all_permutation: list[dict[str, Any]] = []
    for sampler_index, sampler in enumerate(samplers):
        logger.log("sampler_started", sampler=sampler)
        source_features, labels = load_sampler_arrays(source_dir, sampler)
        if args.smoke:
            source_features, labels = select_smoke_rows(
                source_features, labels, int(cfg["smoke"]["max_rows_per_split"]), seed + sampler_index * 100
            )
        tabular_dimensions = int(cfg["input"]["tabular_dimensions"])
        pca_batch = int(cfg["smoke"]["pca_batch_size"] if args.smoke else cfg["pca"]["batch_size"])
        pca_started = time.time()
        pca = fit_incremental_pca(
            source_features["train"],
            tabular_dimensions,
            int(cfg["pca"]["n_components"]),
            pca_batch,
            bool(cfg["pca"].get("whiten", False)),
        )
        feature_dir = run_dir / "features" / sampler
        model_dir = run_dir / "models" / sampler
        feature_dir.mkdir(parents=True, exist_ok=True)
        model_dir.mkdir(parents=True, exist_ok=True)
        joblib.dump(pca, model_dir / "incremental_pca_128d_to_10d.joblib")
        np.savez_compressed(
            model_dir / "pca_parameters.npz",
            components=pca.components_,
            mean=pca.mean_,
            explained_variance=pca.explained_variance_,
            explained_variance_ratio=pca.explained_variance_ratio_,
            singular_values=pca.singular_values_,
        )
        orthonormal_error = float(np.max(np.abs(pca.components_ @ pca.components_.T - np.eye(pca.n_components_))))
        pca_summary = {
            "sampler": sampler,
            "fit_split": "train_only",
            "input_dimensions": int(cfg["input"]["embedding_dimensions"]),
            "output_dimensions": int(pca.n_components_),
            "algorithm": "IncrementalPCA",
            "whiten": bool(cfg["pca"].get("whiten", False)),
            "batch_size": pca_batch,
            "rows_fit": len(source_features["train"]),
            "explained_variance_ratio": pca.explained_variance_ratio_.tolist(),
            "cumulative_explained_variance": np.cumsum(pca.explained_variance_ratio_).tolist(),
            "orthonormality_max_abs_error": orthonormal_error,
            "fit_seconds": time.time() - pca_started,
        }
        write_json(run_dir / "history" / sampler / "pca_summary.json", pca_summary)
        logger.log("pca_fitted", sampler=sampler, cumulative_variance=pca_summary["cumulative_explained_variance"][-1])

        pca_paths, hybrid_paths = {}, {}
        transform_batch = int(cfg["pca"]["transform_batch_size"])
        for split in SPLITS:
            pca_paths[split] = transform_embedding_to_file(
                pca, source_features[split], tabular_dimensions,
                feature_dir / f"{split}_gnn_pca_10d.npy", transform_batch,
            )
            hybrid_paths[split] = create_hybrid_20d_file(
                source_features[split], pca_paths[split], tabular_dimensions,
                feature_dir / f"{split}_hybrid_20d.npy", transform_batch,
            )
            np.save(feature_dir / f"{split}_labels.npy", np.asarray(labels[split], dtype=np.int8))
        write_json(
            feature_dir / "feature_contract.json",
            {
                "source": str((source_dir / "features" / sampler).resolve()),
                "pca_fit_split": "train_only",
                "tabular_dimensions": 10,
                "source_embedding_dimensions": 128,
                "pca_dimensions": 10,
                "hybrid_dimensions": 20,
                "dtype": "float32",
                "rows": {split: len(labels[split]) for split in SPLITS},
            },
        )

        pca_matrices = {split: np.load(pca_paths[split], mmap_mode="r") for split in SPLITS}
        hybrid_matrices = {split: np.load(hybrid_paths[split], mmap_mode="r") for split in SPLITS}
        representation_matrices = {
            "tabular_10d": {split: source_features[split][:, :tabular_dimensions] for split in SPLITS},
            "gnn_pca_10d": pca_matrices,
            "hybrid_20d": hybrid_matrices,
        }
        sampler_probabilities: dict[str, dict[str, np.ndarray]] = {}
        hybrid_model = None
        for representation, matrices in representation_matrices.items():
            model, result, probabilities = train_and_evaluate(
                sampler, representation, matrices, labels, cfg,
                seed + sampler_index * 10, args.smoke, run_dir,
            )
            sampler_probabilities[representation] = probabilities
            all_results.append(result)
            all_rows.append(result_row(result))
            write_json(run_dir / "results" / sampler / f"{representation}.json", result)
            logger.log(
                "classifier_completed", sampler=sampler, representation=representation,
                test_f1=result["test"]["f1"], test_auprc=result["test"]["auprc"],
            )
            if representation == "hybrid_20d":
                hybrid_model = model
        predictions_dir = run_dir / "predictions"
        predictions_dir.mkdir(parents=True, exist_ok=True)
        prediction_payload: dict[str, np.ndarray] = {
            "validation_labels": np.asarray(labels["validation"], dtype=np.int8),
            "test_labels": np.asarray(labels["test"], dtype=np.int8),
        }
        for representation, values in sampler_probabilities.items():
            prediction_payload[f"{representation}_validation_probabilities"] = values["validation"]
            prediction_payload[f"{representation}_test_probabilities"] = values["test"]
            prediction_payload[f"{representation}_optimal_threshold"] = values["threshold"]
        np.savez_compressed(
            predictions_dir / f"{sampler}.npz",
            **prediction_payload,
        )
        if hybrid_model is None:
            raise RuntimeError("Model hybrid tidak terbentuk")
        hybrid_threshold = float(sampler_probabilities["hybrid_20d"]["threshold"])
        repeats = int(
            cfg["smoke"]["permutation_repeats"] if args.smoke else cfg["permutation_importance"]["repeats"]
        )
        records = calculate_block_permutation_importance(
            hybrid_model,
            hybrid_matrices["test"],
            np.asarray(labels["test"]),
            hybrid_threshold,
            sampler,
            repeats,
            evaluation_seed + sampler_index * 10_000,
        )
        all_permutation.extend(records)
        permutation_dir = run_dir / "permutation"
        permutation_dir.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(records).to_csv(permutation_dir / f"{sampler}_repeats.csv", index=False)
        logger.log("sampler_completed", sampler=sampler, permutation_repeats=repeats)
        del source_features, labels, pca_matrices, hybrid_matrices, representation_matrices, hybrid_model
        gc.collect()

    comparison = pd.DataFrame(all_rows)
    comparison.to_csv(run_dir / "equal_dimension_comparison.csv", index=False)
    permutation_frame = pd.DataFrame(all_permutation)
    permutation_frame.to_csv(run_dir / "block_permutation_importance_repeats.csv", index=False)
    permutation_summary = (
        permutation_frame.groupby(["sampler", "block"])[["drop_auprc", "drop_f1"]]
        .agg(["mean", "std", "median", "min", "max"])
        .reset_index()
    )
    permutation_summary.columns = [
        "_".join(str(part) for part in column if part).rstrip("_")
        if isinstance(column, tuple) else str(column)
        for column in permutation_summary.columns
    ]
    permutation_summary.to_csv(run_dir / "block_permutation_importance_summary.csv", index=False)
    write_json(run_dir / "all_results.json", all_results)
    write_summary(run_dir, comparison, permutation_frame)
    create_all_visualizations(run_dir)

    manifest.update(
        {
            "status": "complete",
            "completed_at_utc": datetime.now(timezone.utc).isoformat(),
            "runtime_seconds": time.time() - started,
            "test_evaluations_per_model": 1,
            "pca_fit_split": "train_only",
            "samplers_completed": samplers,
        }
    )
    write_json(run_dir / "manifest.json", manifest)
    logger.log("run_completed", runtime_seconds=manifest["runtime_seconds"])
    build_artifact_index(run_dir)
    return run_dir


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="EXP21D-v8 Fair Control 1B")
    parser.add_argument("--config", default="config.reference.yaml")
    parser.add_argument("--seed", type=int)
    parser.add_argument("--run-id")
    parser.add_argument("--smoke", action="store_true")
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    try:
        run_dir = run_experiment(args)
        print(f"Run selesai: {run_dir}")
    except Exception as error:
        run_dir = getattr(args, "_active_run_dir", None)
        if run_dir:
            manifest_path = Path(run_dir) / "manifest.json"
            manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}
            manifest.update(
                {
                    "status": "failed",
                    "failed_at_utc": datetime.now(timezone.utc).isoformat(),
                    "error": repr(error),
                }
            )
            write_json(manifest_path, manifest)
            RunLogger(Path(run_dir) / "events.jsonl").log(
                "run_failed", error=repr(error), traceback=traceback.format_exc()
            )
            build_artifact_index(run_dir)
        raise


if __name__ == "__main__":
    main()


In [ ]:
%%writefile reanalyze.py
from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import pandas as pd

from artifacts import write_json
from metrics import evaluate_predictions, threshold_curve


REPRESENTATIONS = ("tabular_10d", "gnn_pca_10d", "hybrid_20d")


def reanalyze(run_dir: Path) -> Path:
    rows = []
    curves = {}
    for archive_path in sorted((run_dir / "predictions").glob("*.npz")):
        sampler = archive_path.stem
        archive = np.load(archive_path)
        labels = archive["test_labels"]
        curves[sampler] = {}
        for representation in REPRESENTATIONS:
            probabilities = archive[f"{representation}_test_probabilities"]
            threshold = float(archive[f"{representation}_optimal_threshold"])
            metrics = evaluate_predictions(probabilities, labels, threshold)
            rows.append({"sampler": sampler, "representation": representation, **metrics})
            curves[sampler][representation] = threshold_curve(probabilities, labels)
    if not rows:
        raise FileNotFoundError(f"Arsip prediksi tidak ditemukan di {run_dir / 'predictions'}")
    output = run_dir / "reanalysis"
    output.mkdir(parents=True, exist_ok=True)
    pd.json_normalize(rows).to_csv(output / "metrics_recomputed.csv", index=False)
    write_json(output / "metrics_recomputed.json", rows)
    write_json(output / "threshold_curves_recomputed.json", curves)
    return output


def main() -> None:
    parser = argparse.ArgumentParser(description="Hitung ulang metrik EXP 1B tanpa training")
    parser.add_argument("run_dir", type=Path)
    args = parser.parse_args()
    print(reanalyze(args.run_dir.resolve()))


if __name__ == "__main__":
    main()



In [ ]:
%%writefile validate_implementation.py
from __future__ import annotations

import gc
import json
import tempfile
from pathlib import Path

import numpy as np

from config import load_config
from main import train_and_evaluate, verify_cross_sampler_invariants
from pca_pipeline import (
    SPLITS,
    create_hybrid_20d_file,
    fit_incremental_pca,
    permuted_block,
    transform_embedding_to_file,
    validate_source_contract,
)


def build_fixture(root: Path) -> tuple[dict[str, np.ndarray], dict[str, np.ndarray]]:
    root.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(42)
    sizes = {"train": 320, "validation": 120, "test": 140}
    shared_tabular, shared_labels = {}, {}
    for split, size in sizes.items():
        tabular = rng.normal(size=(size, 10)).astype(np.float32)
        score = tabular[:, 0] + 0.7 * tabular[:, 1] + rng.normal(scale=0.6, size=size)
        labels = (score > np.quantile(score, 0.72)).astype(np.int8)
        shared_tabular[split], shared_labels[split] = tabular, labels
    (root / "exp21d_v5_master_comparison.csv").write_text("sampler,test_f1\nuniform,0.9\n", encoding="utf-8")
    for sampler_index, sampler in enumerate(("uniform", "topology", "importance")):
        feature_dir = root / "features" / sampler
        feature_dir.mkdir(parents=True)
        contract = {
            "tabular_dimensions": 10,
            "graphsage_embedding_dimensions": 128,
            "hybrid_dimensions": 138,
            "split_point": 10,
            "dtype": "float32",
        }
        (feature_dir / "feature_contract.json").write_text(json.dumps(contract), encoding="utf-8")
        for split in SPLITS:
            embedding = rng.normal(size=(len(shared_labels[split]), 128)).astype(np.float32)
            embedding[:, :3] += shared_labels[split][:, None] * (0.8 + sampler_index * 0.1)
            hybrid = np.concatenate([shared_tabular[split], embedding], axis=1)
            np.save(feature_dir / f"{split}_hybrid_features.npy", hybrid)
            np.save(feature_dir / f"{split}_labels.npy", shared_labels[split])
    return shared_tabular, shared_labels


def validate() -> list[str]:
    checks: list[str] = []
    source_dir = Path(__file__).resolve().parent
    config = load_config(source_dir / "config.reference.yaml")
    checks.append("reference_config_valid")
    if (source_dir / "kernel-metadata.json").exists():
        raise AssertionError("kernel-metadata.json tidak boleh ada sebelum pengguna meminta push")
    checks.append("no_kernel_metadata")
    with tempfile.TemporaryDirectory() as temporary:
        root = Path(temporary)
        _, labels = build_fixture(root / "source")
        source = root / "source"
        contracts = validate_source_contract(source, ["uniform", "topology", "importance"], 138, 10, 128)
        if set(contracts) != {"uniform", "topology", "importance"}:
            raise AssertionError("Validasi source contract tidak lengkap")
        checks.append("source_contracts_valid")
        audit = verify_cross_sampler_invariants(source, ["uniform", "topology", "importance"], chunk_size=64)
        if audit["status"] != "passed":
            raise AssertionError("Cross-sampler invariant gagal")
        checks.append("cross_sampler_tabular_and_labels_equal")

        feature_dir = source / "features" / "uniform"
        arrays = {split: np.load(feature_dir / f"{split}_hybrid_features.npy", mmap_mode="r") for split in SPLITS}
        target = {split: np.load(feature_dir / f"{split}_labels.npy", mmap_mode="r") for split in SPLITS}
        pca = fit_incremental_pca(arrays["train"], 10, 10, 64, False)
        identity_error = np.max(np.abs(pca.components_ @ pca.components_.T - np.eye(10)))
        if identity_error > 1e-4:
            raise AssertionError(f"Komponen PCA tidak orthonormal: {identity_error}")
        checks.append("pca_components_orthonormal")
        if pca.n_samples_seen_ != len(arrays["train"]):
            raise AssertionError("PCA tidak fit tepat pada seluruh train")
        checks.append("pca_fit_train_rows_only")

        pca_paths, hybrid_paths = {}, {}
        for split in SPLITS:
            pca_paths[split] = transform_embedding_to_file(
                pca, arrays[split], 10, root / f"{split}_pca.npy", 50
            )
            hybrid_paths[split] = create_hybrid_20d_file(
                arrays[split], pca_paths[split], 10, root / f"{split}_hybrid.npy", 50
            )
            if np.load(pca_paths[split], mmap_mode="r").shape != (len(arrays[split]), 10):
                raise AssertionError("Shape PCA salah")
            if np.load(hybrid_paths[split], mmap_mode="r").shape != (len(arrays[split]), 20):
                raise AssertionError("Shape hybrid 20-d salah")
        checks.append("transformed_shapes_are_10d_and_20d")

        matrix = np.load(hybrid_paths["test"], mmap_mode="r")
        permutation = np.arange(len(matrix))[::-1]
        changed = permuted_block(matrix, 10, 20, permutation)
        if not np.array_equal(changed[:, :10], matrix[:, :10]):
            raise AssertionError("Permutation blok GNN mengubah blok tabular")
        if not np.array_equal(changed[:, 10:], matrix[permutation, 10:]):
            raise AssertionError("Permutation tidak mengacak blok secara joint")
        checks.append("joint_block_permutation_preserves_other_block")

        matrices = {split: np.load(hybrid_paths[split], mmap_mode="r") for split in SPLITS}
        run_dir = root / "run"
        run_dir.mkdir()
        model, result, probabilities = train_and_evaluate(
            "uniform", "hybrid_20d", matrices, target, config, 42, True, run_dir
        )
        if probabilities["validation"].shape != (len(target["validation"]),):
            raise AssertionError("Probabilitas validation tidak lengkap")
        if probabilities["test"].shape != (len(target["test"]),):
            raise AssertionError("Probabilitas test tidak lengkap")
        if result["dimensions"] != 20 or not (0.0 <= result["test"]["auprc"] <= 1.0):
            raise AssertionError("Evaluasi end-to-end tidak valid")
        if not (run_dir / "models" / "uniform" / "xgboost_hybrid_20d.json").is_file():
            raise AssertionError("Model XGBoost tidak tersimpan")
        del model
        checks.append("xgboost_validation_threshold_and_test_evaluation_work")
        del arrays, target, matrices, matrix, changed, pca
        gc.collect()

    main_source = (source_dir / "main.py").read_text(encoding="utf-8")
    if "fit_incremental_pca(\n            source_features[\"train\"]" not in main_source:
        raise AssertionError("Main tidak menunjukkan fit PCA train-only")
    if "test_evaluations_per_model\": 1" not in main_source:
        raise AssertionError("Kontrak single test evaluation hilang")
    checks.append("static_train_only_and_single_test_contract")
    return checks


if __name__ == "__main__":
    for check in validate():
        print(f"[PASS] {check}")



In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)


In [ ]:
!python validate_implementation.py


In [ ]:
!python main.py --config config.kaggle.yaml --seed 42
